**Conformal Predictive Distributions. How to predict full probability distribution Conformal Predictive Distributions**

Author: [Valeriy Manokhin](https://www.linkedin.com/in/valeriy-manokhin-phd-mba-cqf-704731236/). For more resources on Conformal Prediction check out my repo [Awesome Conformal Prediction](https://github.com/valeman/awesome-conformal-prediction)

In this notebook, we demonstrate how [Conformal Predictive Distributions](https://proceedings.mlr.press/v91/vovk18a.html) can produce the full CFD (Cumulative Distribution Function) for each house price prediction using [Crepes](https://github.com/henrikbostrom/crepes) Python package.

You can read more about Conformal Predictive Distributions in my Medium article [How to predict full probability distribution using machine learning Conformal Predictive Distributions](https://medium.com/@valeman/how-to-predict-full-probability-distribution-using-machine-learning-conformal-predictive-f8f4d805e420). 

TL;DR Conformal Predictive Distributions are a powerful method from Conformal Prediction framework. CPDs can produce the whole distribution curve for each of the test point and can be combined with any machine learning or statistical model, are data distribution agnostic and are guaranteed to output valid probabilistic predictions (lack of bias) regardless of the dataset size. 

In this notebook we take LightAutoML created by Kaggle Grandmaster [Alexander Ryzhkov](https://www.kaggle.com/alexryzhkov) and demonstrate how excellent point predictions produced by LightAutoML can be combined with Conformal Predictive Distributions to output the whole probabilistic prediction curve for each point prediction.

We obtain great results using LightAutoML with very little effort and make a submission using both LightAutoML(LAMA) and Conformal Predictive Distributions (CPD). We demonstrate that median predictions obtained from LAMA + CPD combination result in another good prediction. Whilst we only use point predictions from CPD (as this competition is about point predictions), we can generate any point(s) in CDF for any application that requires probabilistic predictions.

In [ ]:
!pip install -U lightautoml
!pip install crepes

**Step 0.1. Import necessary libraries**

In [ ]:
# Standard python libraries
import os
import time
import re
import torch

# Essential DS libraries
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

from crepes import ConformalRegressor, ConformalPredictiveSystem

from crepes.fillings import binning

%matplotlib inline

# LightAutoML presets, task and report generation
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task
from lightautoml.report.report_deco import ReportDeco

Imported models setup For better reproducibility fix numpy random seed with max number of threads for Torch (which usually try to use all the threads on server):

In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Step 0.2. Parameters**

Here we setup the constants to use in the kernel:

* N_THREADS - number of vCPUs for LightAutoML model creation
* N_FOLDS - number of folds in LightAutoML inner CV
* RANDOM_STATE - random seed for better reproducibility
* TEST_SIZE - houldout data part size
* TIMEOUT - limit in seconds for model to train
* TARGET_NAME - target column name in dataset

In [ ]:
N_THREADS = 4 # threads count for lgbm and linear models
N_FOLDS = 5 # folds count for AutoML
RANDOM_STATE = 42 # fixed random state for various reasons
TEST_SIZE = 0.2 # Test size for metric check
TIMEOUT = 7200 # Time in seconds for automl run

**Step 0.4. Data load¶**

* longitude A measure of how far west a house is; a more negative value is farther west. Longitude values range from -180 to +180
* latitude A measure of how far north a house is; a higher value is farther north. Latitude values range from -90 to +90
* housingMedianAge Median age of a house within a block; a lower number is a newer building
* totalRooms Total number of rooms within a block
* totalBedrooms Total number of bedrooms within a block
* population Total number of people residing within a block
* households Total number of households, a group of people residing within a home unit, for a block
* medianIncome Median income for households within a block of houses (measured in tens of thousands of US Dollars)	
* medianHouseValue Median house value for households within a block (measured in US Dollars)

In [ ]:
original = fetch_california_housing()
original_data_df = pd.DataFrame(original["data"], columns=original["feature_names"])
original_data_df["MedHouseVal"] = original["target"]

original_data_df["is_generated"] = False

In [ ]:
original_data_df.head()

In [ ]:
original_data_df.isnull().sum()

In [ ]:
original_data_df.shape

In [ ]:
%%time

train_data_df = pd.read_csv('../input/playground-series-s3e1/train.csv', index_col=0)
train_data_df.head()

In [ ]:
train_data_df.shape

In [ ]:
test_data_df = pd.read_csv('../input/playground-series-s3e1/test.csv', index_col=0)
test_data_df.head()

In [ ]:
train_data_df["is_generated"] = True
test_data_df["is_generated"] = True

In [ ]:
train_data_df.head()

In [ ]:
test_data_df.head()

In [ ]:
submission_df = pd.read_csv('../input/playground-series-s3e1/sample_submission.csv')
submission_df.head()

**Step 0.5. Data splitting for train-test**

In [ ]:
train_data = pd.concat([train_data_df, original_data_df], axis = 0)

In [ ]:
train_data.head()

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data.shape

In [ ]:
tr_data, te_data = train_test_split(train_data, 
                                     test_size=TEST_SIZE,
                                     random_state=RANDOM_STATE)
print('Data splitted. Parts sizes: tr_data = {}, te_data = {}'.format(tr_data.shape, te_data.shape))

In [ ]:
tr_data.head()

**Step 0.6. AutoML preset**

In [ ]:
#create task
%time
task = Task('reg', loss = 'mse', metric = 'mse')

In [ ]:
#setup column roles 
%time
roles = {
    'target': 'MedHouseVal'
}

In [ ]:
# Create AutoML from preset and train on 80% of data
%time 

automl = TabularUtilizedAutoML(task = task, 
                       timeout = 600, # 600 seconds = 10 minutes
                       cpu_limit = 4, # Optimal for Kaggle kernels
                       general_params = {'use_algos': [['linear_l2', 
                                         'lgb', 'lgb_tuned', 'catboost', 'catboost_tuned']]})

In [ ]:
tr_data.head()

In [ ]:
%%time

tr_pred = automl.fit_predict(tr_data, roles=roles, verbose=1)
print(f'Prediction for tr_data:\n{tr_pred}\nShape = {tr_pred.shape}')


In [ ]:
%%time

te_pred = automl.predict(te_data)
print(f'Prediction for te_data:\n{te_pred}\nShape = {te_pred.shape}')

In [ ]:
# Check scores for current predict and aggregated one
rmse_train = mean_squared_error(tr_data['MedHouseVal'].values, tr_pred.data[:, 0], squared=False)
print('RMSE on the train set: {}'.format(rmse_train))

In [ ]:
# Check scores for current predict and aggregated one
rmse_test = mean_squared_error(te_data['MedHouseVal'].values, te_pred.data[:, 0], squared=False)
print('RMSE on the test set: {}'.format(rmse_test))

In [ ]:
%%time
train_pred = automl.fit_predict(train_data, roles = roles, verbose=1)

In [ ]:
# Check scores for current predict and aggregated one
rmse_train = mean_squared_error(train_data['MedHouseVal'].values, train_pred.data[:, 0], squared=False)
print('RMSE on the train set: {}'.format(rmse_train))

In [ ]:
%%time

point_predictions = automl.predict(test_data_df)
y_hat_test = point_predictions.data[:, 0]

In [ ]:
submission_df['MedHouseVal'] = y_hat_test

In [ ]:
submission_df.set_index('id', inplace = True)
submission_df.head()

In [ ]:
submission_df.to_csv('submission_point_predictions.csv')

***Conformal Predictive Distributions***

In [ ]:
# split data into proper training, calibration and test set
tr_data, te_data = train_test_split(train_data, 
                                     test_size=0.1,
                                     random_state=RANDOM_STATE)
print('Data splitted. Parts sizes: tr_data = {}, te_data = {}'.format(tr_data.shape, te_data.shape))

proper_tr_data, cal_data = train_test_split(tr_data, 
                                     test_size=0.1,
                                     random_state=RANDOM_STATE)
print('Data splitted. Parts sizes: proper_tr_data = {}, cal_data = {}, test_data = {}'.format(proper_tr_data.shape, cal_data.shape, te_data.shape))

In [ ]:
%%time
# train the LightAutoMLmodel on the proper training set

proper_tr_pred = automl.fit_predict(proper_tr_data, roles=roles, verbose=1)
print(f'Prediction for proper_tr_data:\n{proper_tr_data}\nShape = {proper_tr_data.shape}')

In [ ]:
%%time
cal_data_predictions = automl.predict(cal_data)

y_hat_cal = cal_data_predictions.data[:, 0]
y_cal = cal_data['MedHouseVal']

residuals_cal = y_cal - y_hat_cal

In [ ]:
cps_std = ConformalPredictiveSystem().fit(residuals=residuals_cal)

In [ ]:
bins_cal, bin_thresholds = binning(values=y_hat_cal, bins=5)

In [ ]:
bins_test = binning(values=y_hat_test, bins=bin_thresholds)

In [ ]:
medians = cps_std.predict(y_hat=y_hat_test,bins=bins_test,lower_percentiles=50, higher_percentiles=50)

In [ ]:
%%time
submission_df['MedHouseVal'] = np.mean(medians, axis=1)
submission_df.to_csv('submission_conformal_predictive_distributions.csv')